Instructions for preparation. Assuming you have python3 installed.

```sh
$ python3 -m venv .venv
$ . ./.venv/bin/activate
```

# Simulation of batched queues and the estimated number of retries.

I choose to simulate batched queues, as we're not expecting Poisson like traffic as input, but rather in a Timed system, messages are arriving at regular intervals, say 40ms. The hardest part about the simulation is estimating what is the maximum number of messages per thread that need to be processed by the loop?

In this case, we have 8 threads, with 8 messages for all threads pending to be inserted into a buffer atomically.

In [3]:
messages = [20, 20, 20, 20, 20, 20, 20, 20]
cores = len(messages)

# Time after a message is processed that the queue is not ready
# for processing
loop_time = 100
loop_outside = 200

# So for example, if it takes only 100ns to process in the loop,
# which is typical for a load, increment pointer and perhaps
# wrap around, and then store, the time outside the loop is 1000ns,
# the dead time is 10.
deadtime = int((loop_outside) / loop_time)

# The simulation uses a time slice of `loop_time` to easily track
# the number of retries.
#
# If you set the `loop_time` to be greater than `loop_outside`, then
# this simulation assumes that the time outside, `deadtime`, is zero.

Once we've defined the batch, we simulate the data being processed. We treat each processing as a single unit of time. We assume that in this time of processing all units there are no more messages arriving, as the processing time is expected to be small, relative to the time outside of the loop processing the Compare and Swap.

In [4]:
import random

total_messages = sum(messages)
messaging = True

core_retries = cores * [0]
core_retries_max = cores * [0]

deadtime_cores = cores * [0]
while (messaging):
    can_process = False
    for c in range(0, cores):
        if messages[c] > 0 and deadtime_cores[c] == 0:
            can_process = True

    if can_process:
        valid_core = False
        while (not valid_core):
            core = random.randint(0, cores - 1)
            valid_core = messages[core] > 0 and deadtime_cores[core] == 0

        # The 'core' won the lottery and will be processed.
        # All other cores with at least one message, must now
        # repeat their loop
        for c in range(0, cores):
            if messages[c] > 0:
                if core == c:
                    core_retries[c] = 0
                else:
                    if deadtime_cores[c] == 0:
                        core_retries[c] = core_retries[c] + 1
                if core_retries[c] > core_retries_max[c]:
                    core_retries_max[c] = core_retries[c]

    print(f"{total_messages:5} ", end="")
    print("".join(f"{m:5}" for m in messages), end="")

    if can_process:
        print(f"  processing on {core:2}", end="")
    else:
        print("  processing on --", end="")

    print("  time before entering loop again: ", end="")
    print("".join(f"{t:5}" for t in deadtime_cores), end="")
    print("")

    if can_process:
        messages[core] = messages[core] - 1
        deadtime_cores[core] = deadtime + 1

    for c in range(0, cores):
        if deadtime_cores[c] > 0:
            deadtime_cores[c] = deadtime_cores[c] - 1

    total_messages = sum(messages)
    messaging = total_messages > 0

print("Maximum retries over all cores:")
print("".join(f"{r:5}" for r in core_retries_max))


  160    20   20   20   20   20   20   20   20  processing on  1  time before entering loop again:     0    0    0    0    0    0    0    0
  159    20   19   20   20   20   20   20   20  processing on  7  time before entering loop again:     0    2    0    0    0    0    0    0
  158    20   19   20   20   20   20   20   19  processing on  2  time before entering loop again:     0    1    0    0    0    0    0    2
  157    20   19   19   20   20   20   20   19  processing on  4  time before entering loop again:     0    0    2    0    0    0    0    1
  156    20   19   19   20   19   20   20   19  processing on  7  time before entering loop again:     0    0    1    0    2    0    0    0
  155    20   19   19   20   19   20   20   18  processing on  3  time before entering loop again:     0    0    0    0    1    0    0    2
  154    20   19   19   19   19   20   20   18  processing on  4  time before entering loop again:     0    0    0    2    0    0    0    1
  153    20   19   1

This simulation shows, if the `deadtime` is zero, there can be contention, and if the number of retries should be minimised there are two ways:

* there needs to be some kind of prioritisation when processing the loop, so instead of being random, and serialised by which ever thread wins in the CAS, something serialises this; or
* reduce the amount of work to be done in the loop, so that it is faster, relative to the time outside of the loop; or
* add artificial delays outside of the loop, to reduce the probability of contention

Everywhere in the table above, there is `processing on --`, this means that no core is executing inside the loop.

We can easily see, that if the time to process a loop, and outside the loop, remains constant, the likelihood of retries increases as the number of cores increase due to contention. What this means, when the `deadtime` is small enough that `N`*`loop_time` > `deadtime` for N cores holds, and which core runs is non-deterministic, it is *usually* half of the number of the messages to be processed.